# Spark transformations
## What do we mean by transformations?
![transformations](images/transformations.png)
- In Spark we read the data from a data source and create one of the two things. 
    - DataFrames
        - The DataFrame is the programmatic interface for your data
        - Transformation : Here the programmatic approach is implemented when it comes to transformations
    - Database table
        - The Database table is the sql interface for your data.
        - Transformation : Here the sql approach is implemented when it comes to transformations
- Both the database tables and the dataframes are the same but two different interfaces.
-  Transformation is nothing but:
    - Combining DataFrames
    - Aggregating and Summarizing 
    - Applying functions and built-in transformations
    - Using built-in and column-level functions
    - Creating and using UDFs
    - Creating column expressions
    - Referencing rows/columns
### Working with DataFrame rows:
- Spark dataframe is a dataset of rows.
- Each row in the dataFrame is a single record represented by an object of type row.
- Most of the time we do not directly work with the entire row.
- However there are three specific senarios where we have to directly work with the row object.
    - Manually creating rows and dataFrame.
    - Collecting DataFrame rows to the driver.
    - Work with an individual row in spark transformations.
#### Example 1: 
- Here in this example you will see where creating a dataFrame manually on the fly helps in unit testing of the functions and method that you create
- Why creating dataFrame helps manually instead of importing sample data from a csv file?
    - Everytime we have to test a function or a method its not possible to read the csv file and bring in some sample data to create a dataFrame because it will make testing the application significantly slower due to un-necessary I/O overhead.
- In this example I have written a function that converts dataType from string to date type given the column name in a dataFrame
- I have also written a test case using python's built in unit test tool to simulate real world senario to the best of my ability.
#### Here is the final code 
#### dataframe_transformations.py
```python
from pyspark.sql.functions import to_date
class DataFrameTransformations:
    def __init__(self,spark):
        self.spark_object = spark
    def count_by_country(self,spark_df):
        intermediate_result_df = (
                spark_df
                .where("CallType is not null")
                .select("CallType","Zipcode")
                .groupby("CallType","Zipcode")
            )
        row_count = intermediate_result_df.count()
        result_df = row_count.orderBy("count",ascending=False)
        return result_df
    
    """This methods onverts the column with dates in string datatype to date datatype"""
    def convert_to_date_type(self, spark_df, date_format, col_name):
        """Converts a string column to date using the given format."""
        return spark_df.withColumn(col_name, to_date(col_name, date_format))
```
#### unit_test.py
```python
import unittest
import os 
import sys
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(os.path.dirname(CURRENT_DIR))  # one level higher
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from pyspark.sql import SparkSession, Row
from SparkDFTransformations.transformations.dataframe_transformations import DataFrameTransformations
from datetime import date


class TestDataFrameTransformations(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.spark = (
            SparkSession
            .builder
            .appName("PySparkUnitTest")
            .master("local[2]")
            .getOrCreate()
        )
        cls.transformer = DataFrameTransformations(cls.spark)

    @classmethod
    def tearDownClass(cls):
        cls.spark.stop()

    def test_convert_to_date_type(self):
        # Sample test DataFrame
        data = [
            Row(id="1", EventDate="3/11/2025"),
            Row(id="2", EventDate="4/11/2025")
        ]

        schema = "id STRING, EventDate STRING"
        df = self.spark.createDataFrame(data, schema)

        # Apply transformation
        result_df = self.transformer.convert_to_date_type(df, "d/M/yyyy", "EventDate")

        # Collect and check types
        result = result_df.collect()

        # Assert that EventDate is converted to a Python date object
        self.assertIsInstance(result[0]['EventDate'], date)
        self.assertEqual(result[0]['EventDate'], date(2025, 11, 3))

        print("✅ convert_to_date_type() test passed!")


if __name__ == '__main__':
    unittest.main()
```
#### Example 2: Ingest unstructured data from an apache.log file and perform dataFrame transformations on it.
- Here in this example I am going to simulate how to handle unstructured data in pyspark using a log_file from a server
- The data file is an apache webserver log file.
- We do have some pattern in this log file but this file is not even a semi-structure data file it is just a log dump
- So If I try to read this file into a dataFrame all I am going to get is a single row of strings
- I won't be getting columns in the dataFrame because the log file is an unstructured data file.
- In this case I won't be able to use many of the higher level transformation such as aggregation and grouping.
- We need to find a way to extract some well defined field from the data.
    - The way that I chose to extract data from the apache server log file is to use regex
    - I used this regex ```log_regex = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'``` to collect these information from the log file : 
    ``` bash
        IP
        client
        datetime
        cmd
        request
        protocol
        status
        bytes
        referrer
        userAgent
    ```
    - Example : 
    ```python
    log_regex = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'
    file_df = (
        self.spark_object
        .read
        .format("text")
        .load(file_dir)
    )
    spark_df = file_df.select(regexp_extract())
    ```
    - **Explaiantion :**
        - ```regexp_extract()``` this functionn takes in 3 arguments
            - The first argument is the source string or the field name
            - The second argument is the regular expression
                - This regular expression will extract all 11 fields
            - The third field is the index no of the the field to be selected and extract from the log text
        - Your final code should look something like this :
        ```python
        def import_data_text(self,file_dir,unstructured:bool=False):
        try:
            # This regex is used to extract these data from the log text file
            """
            IP
            client
            datetime
            cmd
            request
            protocol
            status
            bytes
            referrer
            userAgent
            """
            log_regex = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'
            file_df = (
                self.spark_object
                .read
                .format("text")
                .load(file_dir)
            )
            spark_df = file_df.select(
                            regexp_extract('value',log_regex,1).alias("ip"),
                            regexp_extract('value',log_regex,4).alias("date"),
                            regexp_extract('value',log_regex,6).alias("request"),
                            regexp_extract('value',log_regex,10).alias("referrer"),
                        )
            self.log_df_metrics(spark_df=spark_df,file_dir=file_dir)
            return spark_df
        except Exception as e:
            self.logger.error(str(e))
            raise
        ```
- Now I want to group the rows based on referrer column also I want it to be grouped based on the website's domain names instead of the entire url 
    - example : 
    ```bash
    http://semicomplete.com/presentations/logstash-monitorama-2013/
    ```
    to this 
    ```
    http://www.semicomplete.com 
    ```
    - Here is the final code : 
    ```python
    def groupby_referrer(self,spark_df, col_name):
        try:
            if col_name == "referrer":
                result_df = (
                        spark_df
                        .withColumn(col_name,F.substring_index(F.col(col_name),"/",3))
                        .groupBy(col_name)
                        .count()
                    )
            else:
                result_df = (
                    spark_df
                    .groupBy(col_name)
                    .count()
                )
            self.log_df_metrics(result_df,operation_name="groupby_referrer")
            return result_df
        except Exception as e:
            self.logger(str(e))
            raise
    ```
    - Explaiantion : 
        - ```F.col(col_name)``` : 
            - This creates a Spark Column object referring to the given column name — e.g. "referrer".
        - ```F.substring_index(F.col(col_name), "/", 3)```
            - The function substring_index is a string function in PySpark that:
                - Returns the substring from the start of the string up to the n-th occurrence of a delimiter.
                - Remember if you enter ```3``` then only first two substrings in between the delimiter will be considered 
                - str → The column or string you’re operating on.
                - delimiter → The character(s) to split by ("/" here).
                - count → How many parts (delimiters) to include.
                - ```/``` means split by /:
                ```bash
                ['http:', '', 'semicomplete.com', 'presentations', 'logstash-monitorama-2013', '']
                ```
                - ```3``` count = 3:
                ```python
                'http://semicomplete.com'
                ```
        - ```withColumn()```
            - This replaces the referrer column with only its domain-level substring
            - When you later do .groupBy("referrer").count(), Spark groups all hits from the same domain together.
- **FINAL CODE** : You will get the final code in this github repo : https://github.com/aryan68125/Deep-learning-prerequisite/tree/master/pyspark/SparkTransformations/SparkDFTransformations

### Working with DataFrame columns: 
- **What is a column and how to reference it?**
    - Spark dataFrame columns are the objects of type column.
    - These column objects do not make any sense outside the context of the dataFrame and you cannot manipulate them independently.
    - Columns are always use within a spark dataFrame transformations.
    - There are two ways to refer to columns in a dataFrame transformation
        - Column String
            - Column string is the simplest method to access the column
            - Example : 
            ```python
            def select_col(self,spark_df, saprk_df_name:str = "",col_list:list = []):
            try:
                if len(col_list):
                    return spark_df.select(*col_list)
                else:
                    self.logger.error(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
                    raise ValueError(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
            except Exception as e:
                self.logger.error(str(e))
                raise
            ```
        - Column object
- **How to create column expressions?**
    - In Spark you can create a column expression using two types.
        - String expressions or SQL expressions
        ```python
        # defined in transformation.py
        def select_col(self,spark_df, saprk_df_name:str = "",col_list:list = [], expr_str:str=""):
        try:
            if len(col_list) and not expr_str:
                return spark_df.select(*col_list)
            elif len(col_list) and expr_str:
                return spark_df.select(*col_list, F.expr(expr_str))
            else:
                self.logger.error(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
                raise ValueError(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
        except Exception as e:
            self.logger.error(str(e))
            raise

        # usage in main.py file
        spark_df_selected_col = df_t.select_col(spark_df=spark_df_csv,saprk_df_name="spark_df_csv",col_list=["FL_DATE","ORIGIN_CITY_NAME", "DEST_CITY_NAME", "DISTANCE"],expr_str="DISTANCE * 1.609344 as DISTANCE_KM")
        ```
        - Column Object expressions
        **main.py**
        ```python
        from pyspark.sql import SparkSession
        # import related to logging
        from lib.logger import Log4j, LogSparkDataframe
        # import related to custom spark configurations
        from lib.utils import get_spark_app_config
        # imports related to exporting dataframe
        from lib.write_df import ExportSparkDataFrame
        # import writing sparkdf to tables related stuff
        from lib.load_df_data_into_table import LoadSparkDFIntoTable
        # logging related imports 
        import os

        # Imports related to ingest data
        from lib.ingest_data import IngestData
        # Transform data
        from transformations.dataframe_transformations import DataFrameTransformations

        # imports related to cleanup when the main_app.py is re-run
        from lib.clean_up_file_system import CleanupAppFileSystemOnReRun

        if __name__ == "__main__":
            # logging related logic
            # Get the current project's directory
            project_dir = os.path.dirname(os.path.abspath(__file__))
            # cleanup loggic on main_app.py re-run
            # initialize the cleanup class
            cleanup = CleanupAppFileSystemOnReRun(project_dir)
            cleanup.execute_cleanup(clean_logs=True)

            # Get the Log4j.properties file directory
            log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
            # Save the directory where the generated log files must reside
            log_dir = os.path.join(project_dir, "log4j_properties", "logs")
            # Create the directory where the log files must be kept if not present
            os.makedirs(log_dir, exist_ok=True)

            conf = get_spark_app_config()
            spark = (
                SparkSession
                .builder
                .config(conf=conf)
                .config("spark.driver.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .config("spark.executor.extraJavaOptions",
                        f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
                .config("spark.jars.packages", "org.apache.spark:spark-avro_2.13:4.0.1")
                .enableHiveSupport()
                .getOrCreate()
            )

            # initialize logger class 
            logger = Log4j(spark)

            # initialize the spark dataframe logger 
            sp_df_logger = LogSparkDataframe(spark)

            # logging some debug related stuff 
            logger.debug(f"log4j.properties file dir = {log4j_config_path}")
            logger.debug(f"log files dir = {log_dir}")
            logger.debug(f"log dir exists = {os.path.exists(log_dir)}")
            
            logger.info("Reading the data from the directory")
            dataset_dir = os.path.join(project_dir,"dataset")

            # Simulating unstructured data ingested from apache server logs STARTS
            """Import data from a log file (unstructured data) STARTS"""
            # import data from a text file 
            file_name = conf.get("file_name_text")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            ingest_data = IngestData(spark)
            spark_df_text = ingest_data.import_data_text(file_dir=file_dir,unstructured=True)
            # log spark_df_parquet dataframe
            sp_df_logger.log_df(spark_df=spark_df_text,spark_df_name="spark_df_text")
            """Import data from a log file (unstructured data) ENDS"""

            """Data Transformation STARTS"""
            df_t = DataFrameTransformations(spark)

            # conver the string dataType datetime to timestamp datatype datetime
            spark_df_text = df_t.convert_str_to_timestamp_type(spark_df=spark_df_text,col_name="date",spark_df_name="spark_df_text")
            sp_df_logger.log_df(spark_df=spark_df_text,spark_df_name="spark_df_text")

            # groupBy() rows based on referrer column
            spark_df_text = df_t.groupby_referrer(spark_df=spark_df_text, col_name="referrer")
            sp_df_logger.log_df(spark_df=spark_df_text,spark_df_name="spark_df_text")
            """Data Transformation ENDS"""
            # Simulating unstructured data ingested from apache server logs ENDS

            # Working with dataFrame columns STARTS
            """Import data from a paraquet file STARTS"""
            # import data from a text file 
            file_name = conf.get("file_name_csv")
            file_dir = os.path.join(dataset_dir,file_name)
            logger.debug(f"file_name_json dir = {file_dir}")
            ingest_data = IngestData(spark)
            spark_df_csv = ingest_data.import_data_csv(file_dir=file_dir)
            # log spark_df_csv dataframe
            sp_df_logger.log_df(spark_df=spark_df_csv,spark_df_name="spark_df_csv")
            """Import data from a paraquet file ENDS"""

            """Transformation STARTS"""
            df_t = DataFrameTransformations(spark)

            # select column from a dataFrame
            spark_df_bool = df_t.select_col(
                spark_df=spark_df_csv,
                saprk_df_name="spark_df_csv",
                col_list=["FL_DATE", "CANCELLED", "DISTANCE"],
                convert_to_bool_col_name="CANCELLED"
                )
            # log spark_df_csv dataframe
            sp_df_logger.log_df(spark_df=spark_df_bool,spark_df_name="spark_df_selected_col")
            sp_df_logger.log_df_metrics(spark_df=spark_df_bool, spark_df_name="spark_df_selected_col")

            # Convert miles to km
            spark_df_selected_col = df_t.select_col(spark_df=spark_df_csv,saprk_df_name="spark_df_csv",col_list=["FL_DATE","ORIGIN_CITY_NAME", "DEST_CITY_NAME", "DISTANCE"],expr_str="DISTANCE * 1.609344 as DISTANCE_KM")
            # log spark_df_csv dataframe
            sp_df_logger.log_df(spark_df=spark_df_selected_col,spark_df_name="spark_df_selected_col")
            sp_df_logger.log_df_metrics(spark_df=spark_df_selected_col, spark_df_name="spark_df_selected_col")
            
            # Convert CANCELLED column data from integer 0 and 1 to True and False
            spark_df_cancelled_col_bool = df_t.select_col(spark_df=spark_df_csv,saprk_df_name="spark_df_csv",col_list=["FL_DATE","ORIGIN_CITY_NAME", "DEST_CITY_NAME", "DISTANCE","CANCELLED"],convert_to_bool_col_name="CANCELLED")
            # log spark_df_csv dataframe
            sp_df_logger.log_df(spark_df=spark_df_cancelled_col_bool,spark_df_name="spark_df_cancelled_col_bool")
            sp_df_logger.log_df_metrics(spark_df=spark_df_cancelled_col_bool, spark_df_name="spark_df_cancelled_col_bool")
            """Transformation ENDS"""
            # Working with dataFrame columns ENDS
            
            # This line is for debugging only comment after <required to see the partitions of spark dataFrame>
            # input("Please enter")
            spark.stop()
        ```
        **transformation.py**
        ```python
        import os 
        import sys
        CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
        print(f"CURRENT_DIR >> {CURRENT_DIR}")
        PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
        print(f"PROJECT_ROOT >> {PROJECT_ROOT}")
        if PROJECT_ROOT not in sys.path:
            sys.path.insert(0, PROJECT_ROOT)
        print(f"printing sys path of python >>>")
        for p in sys.path[:5]:
            print("  ", p)

        # import transformation related stuff
        from pyspark.sql import functions as F
        from pyspark.sql.types import BooleanType
        from pyspark.sql.functions import udf
        # import logging related stuff
        from lib.logger import Log4j
        from lib.app_monitor import GetDataFrameMemory

        class DataFrameTransformations:
            def __init__(self,spark):
                self.spark_object = spark
                self.logger = Log4j(spark)
                self.metrics = GetDataFrameMemory(spark)


            """This methods onverts the column with dates in string datatype to date datatype"""
            def convert_str_to_timestamp_type(self, spark_df, col_name,spark_df_name):
                try:
                    self.logger.debug(f"converting date columns from string datatype to timestamp datatype in dataFrame {spark_df_name}")
                    # Apply Spark’s to_timestamp() with the exact format
                    if spark_df_name == "spark_df_text" and col_name=="date":
                        parsed_col = F.to_timestamp(F.col(col_name), "dd/MMM/yyyy:HH:mm:ss Z")
                    else:
                        self.logger.error(f"You need to implement the rules to related to dataType conversion to date type from string type")
                        return None

                    result_df = spark_df.withColumn(col_name, parsed_col)
                    self.log_df_metrics(result_df,operation_name="convert_str_to_timestamp_type")
                    return result_df
                except Exception as e:
                    self.logger.error(str(e))
                    raise
            
            """this method groups the data based on referrer"""
            def groupby_referrer(self,spark_df, col_name):
                try:
                    if col_name == "referrer":
                        result_df = (
                                spark_df
                                .where(f"trim({col_name}) != '-'")
                                .withColumn(col_name,F.substring_index(F.col(col_name),"/",3))
                                .groupBy(col_name)
                                .count()
                            )
                    else:
                        result_df = (
                            spark_df
                            .where(f"trim({col_name}) != '-'")
                            .groupBy(col_name)
                            .count()
                        )
                    self.log_df_metrics(result_df,operation_name="groupby_referrer")
                    return result_df
                except Exception as e:
                    self.logger.error(str(e))
                    raise

            """This method will select the columns based on column_name"""
            def select_col(self,spark_df, saprk_df_name:str = "",col_list:list = [], expr_str:str="",convert_to_bool_col_name:str=""):
                try:
                    if len(col_list) and not expr_str and not convert_to_bool_col_name:
                        return spark_df.select(*col_list)
                    elif len(col_list) and expr_str and not convert_to_bool_col_name:
                        return spark_df.select(*col_list, F.expr(expr_str))
                    elif len(col_list) and not expr_str and convert_to_bool_col_name:
                        bool_udf = udf(self.bool_parser, BooleanType())
                        result_df = spark_df.withColumn(
                            convert_to_bool_col_name,
                            bool_udf(F.col(convert_to_bool_col_name))
                        )
                        return result_df.select(*col_list, convert_to_bool_col_name)
                    else:
                        self.logger.error(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
                        raise ValueError(f"column list cannot be empty! you need column names to be able to select columns from {saprk_df_name} dataFrame")
                except Exception as e:
                    self.logger.error(str(e))
                    raise

            # utility methods
            @staticmethod
            def bool_parser(bool_val):
                if bool_val in (1, "1", True):
                    return True
                elif bool_val in (0, "0", False):
                    return False
                else:
                    return None  # handle unexpected cases gracefully

            def log_df_metrics(self,spark_df,operation_name):
                self.logger.info(f"{operation_name} :: The memory taken by the spark dataFrame is = {self.metrics.get_mem_usage(spark_df).get("mem")} MB")
                schema_str = spark_df._jdf.schema().treeString()
                self.logger.debug(f"Spark DataFrame Schema (expanded): {schema_str}")
        ```
        - Explaination:
        - Focus in select_col method 
        ```python
            bool_udf = udf(self.bool_parser, BooleanType())
                            result_df = spark_df.withColumn(
                                convert_to_bool_col_name,
                                bool_udf(F.col(convert_to_bool_col_name))
                            )
        ```
        - This class method (select_col) converts a column containing numeric or string flags like "1", "0" into Boolean (True/False)
        - But Spark doesn’t have built-in logic to handle "1"/"0" or string booleans.
        - So I define our own Python function (bool_parser) and wrap it with Spark’s udf() mechanism.
        - ```bool_udf = udf(self.bool_parser, BooleanType())```
            - ```self.bool_parser``` → This is my Python function where it checks if the value is 1 and replaces it with the truth value and replaces 0th value with the false value
            - ```udf(self.bool_parser, BooleanType())``` wraps that Python function into a Spark UDF that can be applied on DataFrame columns distributedly.
            - BooleanType() tells Spark that the function will return boolean values (True/False).
        - ```spark_df.withColumn()``` 
            - It is used for column transformations in PySpark.
            - Creates a new DataFrame by adding, replacing, or transforming a column.
            - In this case it creates a new DataFrame with the same columns as spark_df, except it replaces or adds one column.
            - ```convert_to_bool_col_name``` the column name you want to transform (e.g., "flag").
            - ```bool_udf(F.col(convert_to_bool_col_name))``` Applies my UDF to every value in that column
            - Essentially, this line tells Spark: For each row, take the column flag, run bool_parser() on it, and store the result back in flag
        - ```return result_df.select(*col_list, convert_to_bool_col_name)``` This simply selects the original columns I asked for (col_list) plus the converted boolean column.